In [0]:

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 50 * 1024 * 1024)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.join.preferSortMergeJoin", "true")
spark.conf.set("spark.sql.shuffle.partitions", "100")

In [0]:

# Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import json
import re

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", StringType(), True),
    StructField("order_approved_at", StringType(), True),
    StructField("order_delivered_carrier_date", StringType(), True),
    StructField("order_delivered_customer_date", StringType(), True),
    StructField("order_estimated_delivery_date", StringType(), True)
])

order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])

In [0]:
orders_raw = spark.read \
    .schema(orders_schema) \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .csv("gs://gds-external-bucket-for-databrick/ecom_data/orders.csv")

order_items_raw = spark.read \
    .schema(order_items_schema) \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .csv("gs://gds-external-bucket-for-databrick/ecom_data/order_items.csv")

customers_raw = spark.read \
    .schema(customers_schema) \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .csv("gs://gds-external-bucket-for-databrick/ecom_data/customers.csv")

In [0]:
order_items_raw.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|   19-09-2017 09:45| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|   03-05-2017 11:05|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|   18-01-2018 14:48|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|   15-08-2018 10:10|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|   13-02-2017 13:57|199.9|        18.14|
+--------------------+-------------+------------

In [0]:
# Order
before_order = orders_raw.count()
orders_raw = orders_raw.dropDuplicates()
after_order = orders_raw.count()

print("Order dropped:", before_order - after_order)

# Order Items
before_items = order_items_raw.count()
order_items_raw = order_items_raw.dropDuplicates()
after_items = order_items_raw.count()

print("Order Items dropped:", before_items - after_items)

# Customers
before_customers = customers_raw.count()
customers_raw = customers_raw.dropDuplicates()
after_customers = customers_raw.count()

print("Customers dropped:", before_customers - after_customers)

Order dropped: 198378
Order Items dropped: 162711
Customers dropped: 99412


In [0]:
from pyspark.sql.functions import *

orders_clean = orders_raw.withColumn(
    "order_ts",
     to_timestamp(col("order_purchase_timestamp"), "dd-MM-yyyy HH:mm")
)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

orders_metrics = orders_raw.select(
    spark_sum(col("customer_id").isNull().cast("int")).alias("null_customer")
).collect()[0]

orders_valid = orders_clean.filter(
    col("customer_id").isNotNull()
)

orders_invalid = orders_clean.filter(
    col("customer_id").isNull()
)

order_items_metrics = order_items_raw.select(
    spark_sum((col("price") <= 0).cast("int")).alias("invalid_price"),
    spark_sum((col("order_item_id") <= 0).cast("int")).alias("invalid_quantity")
).collect()[0]

order_items_valid = order_items_raw.filter(
    (col("price") > 0) &
    (col("order_item_id") > 0)
)

order_items_invalid = order_items_raw.filter(
    (col("price") <= 0) |
    (col("order_item_id") <= 0)
)

print("----- Data Quality Report -----")
print(f"Null customer rows         : {orders_metrics['null_customer']}")
print(f"Invalid price rows         : {order_items_metrics['invalid_price']}")
print(f"Invalid quantity rows      : {order_items_metrics['invalid_quantity']}")

print("\n----- Final Counts -----")
print(f"Valid orders               : {orders_valid.count()}")
print(f"Invalid orders             : {orders_invalid.count()}")
print(f"Valid order items          : {order_items_valid.count()}")
print(f"Invalid order items        : {order_items_invalid.count()}")

----- Data Quality Report -----
Null customer rows         : 99441
Invalid price rows         : 1014
Invalid quantity rows      : 177543

----- Final Counts -----
Valid orders               : 99441
Invalid orders             : 99441
Valid order items          : 112650
Invalid order items        : 178105


In [0]:
# part 3

from pyspark.sql.functions import col, sum, count, to_date, broadcast

order_items_agg = (
    order_items_valid
    .groupBy("order_id")
    .agg(
        sum("price").alias("total_amount"),
        count("*").alias("total_items")
    )
)
order_items_agg.show(5)

+--------------------+------------+-----------+
|            order_id|total_amount|total_items|
+--------------------+------------+-----------+
|0c283c8e0c146f452...|       139.0|          1|
|16e233a1bc2e99095...|        54.9|          1|
|3e6e1b15703081361...|        49.9|          1|
|5000739c56c2fbfbc...|        99.9|          1|
|604614c5012e0fbe1...|        27.9|          1|
+--------------------+------------+-----------+
only showing top 5 rows


In [0]:
orders_fact = (
    orders_valid
    .join(broadcast(order_items_agg), "order_id", "inner")
    .withColumn("order_date", to_date("order_ts"))
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "total_amount",
        "total_items",
        "order_status"
    )
)

orders_fact.show(5, False)

+--------------------------------+--------------------------------+----------+------------+-----------+------------+
|order_id                        |customer_id                     |order_date|total_amount|total_items|order_status|
+--------------------------------+--------------------------------+----------+------------+-----------+------------+
|95266dbfb7e20354baba07964dac78d5|a166da34890074091a942054b36e4265|2018-01-08|129.99      |1          |delivered   |
|634e8f4c0f6744a626f77f39770ac6aa|05e996469a2bf9559c7122b87e156724|2017-08-09|219.0       |1          |delivered   |
|f9f76d0f8ab416a42a2a3dad0d710162|e4a32f8f3648818820a821cd577ccdba|2017-11-30|269.9       |1          |delivered   |
|f04bfdbef5359607d39e66fccc9cc0de|ddbd5d378c4a0981ba1ef29fb8e40b8f|2017-09-13|721.5       |4          |delivered   |
|4709741e829775567b92abc42437b461|bd13582471af02ce718568d9cf8bbd93|2018-03-29|94.5        |1          |delivered   |
+--------------------------------+------------------------------

In [0]:
from pyspark.sql.functions import avg, min, max

customer_metrics = (
    orders_fact
    .groupBy("customer_id")
    .agg(
        count("order_id").alias("total_orders"),
        sum("total_amount").alias("lifetime_value"),
        avg("total_amount").alias("avg_order_value"),
        min("order_date").alias("first_order_date"),
        max("order_date").alias("last_order_date")
    )
)
customer_metrics.show()

+--------------------+------------+--------------+---------------+----------------+---------------+
|         customer_id|total_orders|lifetime_value|avg_order_value|first_order_date|last_order_date|
+--------------------+------------+--------------+---------------+----------------+---------------+
|e2431d4ec21d4a66b...|           1|         28.99|          28.99|      2017-11-29|     2017-11-29|
|df0b1989cacff9ad7...|           1|          65.9|           65.9|      2018-07-23|     2018-07-23|
|72ecfc69f7d90359d...|           1|         110.0|          110.0|      2018-03-13|     2018-03-13|
|ececd83cf61745e93...|           1|          79.9|           79.9|      2018-03-22|     2018-03-22|
|b2f3350f0fcfb2df7...|           1|         107.0|          107.0|      2018-02-14|     2018-02-14|
|f82d1df095bf073a1...|           1|         89.99|          89.99|      2018-07-31|     2018-07-31|
|b2e01250b17d8fca9...|           1|          46.9|           46.9|      2018-08-20|     2018-08-20|


In [0]:
# orders_fact = orders_fact.coalesce(50)


In [0]:
orders_fact = orders_fact.repartition("order_date")

orders_fact.write \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .option("compression", "snappy") \
    .parquet("/gold/orders_fact")

In [0]:
customer_metrics.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet("/gold/customer_metrics")

In [0]:
# part 5

orders_fact.explain(True)

== Parsed Logical Plan ==
'RepartitionByExpression ['order_date]
+- Project [order_id#3719, customer_id#3720, order_date#4714, total_amount#4579, total_items#4580L, order_status#3721]
   +- Project [order_id#3719, customer_id#3720, order_status#3721, order_purchase_timestamp#3722, order_approved_at#3723, order_delivered_carrier_date#3724, order_delivered_customer_date#3725, order_estimated_delivery_date#3726, order_ts#3990, total_amount#4579, total_items#4580L, to_date(order_ts#3990, None, Some(Etc/UTC), true) AS order_date#4714]
      +- Project [order_id#3719, customer_id#3720, order_status#3721, order_purchase_timestamp#3722, order_approved_at#3723, order_delivered_carrier_date#3724, order_delivered_customer_date#3725, order_estimated_delivery_date#3726, order_ts#3990, total_amount#4579, total_items#4580L]
         +- Join Inner, (order_id#3719 = order_id#3727)
            :- Filter isnotnull(customer_id#3720)
            :  +- Project [order_id#3719, customer_id#3720, order_status#

In [0]:
orders_fact.rdd.getNumPartitions()

4

In [0]:
# part 6
orders_stream = (
    spark.readStream
    .schema(orders_schema)
    .option("header", "true")
    .csv("gs://gds-external-bucket-for-databrick/stream/orders")
)


In [0]:
from pyspark.sql.functions import col, to_timestamp

orders_clean_stream = (
    orders_stream
    .filter(col("customer_id").isNotNull())
    .withColumn(
        "order_ts",
        to_timestamp(col("order_purchase_timestamp"), "dd-MM-yyyy HH:mm")
    )
)

In [0]:
spark.conf.get("spark.sql.extensions")

'io.delta.sql.DeltaSparkSessionExtension'

In [0]:
query = (
    orders_clean_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "gs://gds-external-bucket-for-databrick/checkpoints/orders_stream/")
    .trigger(availableNow=True)
    .start("gs://gds-external-bucket-for-databrick/silver/orders_stream/")
)

In [0]:
query.status

{'message': 'Initializing sources',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [0]:
spark.read.format("delta") \
    .load("gs://gds-external-bucket-for-databrick/silver/orders_stream/") \
    .show(5)

+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|           order_ts|
+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|        02-10-2017 10:56| 02-10-2017 11:07|            04-10-2017 19:55|             10-10-2017 21:25|             18-10-2017 00:00|2017-10-02 10:56:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|        24-07-2018 20:41| 26-07-2018 03:24|            26-07-2018 14:31|             07-08-2018 15:27|      